# 模型推理 - 使用 QLoRA 微调后的 ChatGLM3-6B

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

# 定义全局变量和参数
# model_name_or_path = 'THUDM/chatglm3-6b'  # 模型ID或本地路径
model_name_or_path = '/root/autodl-fs/ZhipuAI/chatglm3-6b'
peft_model_path = f"models/demo/{model_name_or_path}"


In [2]:
config = PeftConfig.from_pretrained(peft_model_path)

q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float32)

base_model = AutoModel.from_pretrained(config.base_model_name_or_path,
                                       quantization_config=q_config,
                                       trust_remote_code=True,
                                       device_map='auto')
base_model.requires_grad_(False)
base_model.eval()

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

## 微调前后效果对比

### ChatGLM-6B

```
输入：

类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领

ChatGLM-6B 微调前输出：

* 版型：修身
* 显瘦：True
* 风格：文艺
* 简约：True
* 图案：印花
* 撞色：True
* 裙下摆：直筒或微喇
* 裙长：中长裙
* 连衣裙：True

ChatGLM-6B 微调后输出：

一款简约而不简单的连衣裙，采用撞色的印花点缀，打造文艺气息，简约的圆领，修饰脸型。衣袖和裙摆的压褶，增添设计感，修身的版型，勾勒出窈窕的身材曲线。
```

### ChatGLM2-6B

```
输入：
类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领

微调前：
这款裙子,版型显瘦,采用简约文艺风格,图案为印花和撞色设计,裙下摆为压褶裙摆,裙长为连衣裙,适合各种场合穿着,让你舒适自在。圆领设计,优雅清新,让你在任何场合都充满自信。如果你正在寻找一款舒适、时尚、优雅的裙子,不妨 考虑这款吧!

微调后: 
这款连衣裙简约的设计，撞色印花点缀，丰富了视觉，上身更显时尚。修身的版型，贴合身形，穿着舒适不束缚。圆领的设计，露出精致锁骨，尽显女性优雅气质。下摆压褶的设计，增添立体感，行走间更显飘逸。前短后长的设计，显 得身材比例更加完美。文艺的碎花设计，更显精致。
```

### ChatGLM3-6B

In [3]:
input_text = '类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领'
print(f'输入：\n{input_text}')
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


输入：
类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领


In [4]:
response, history = base_model.chat(tokenizer=tokenizer, query=input_text)
print(f'ChatGLM3-6B 微调前：\n{response}')

/root/miniconda3/lib/python3.10/site-packages/bitsandbytes/nn/modules.py:426: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


ChatGLM3-6B 微调前：
这款连衣裙以简约风格为主，采用了撞色和印花图案，让整个裙子看起来更有层次感。同时，裙子的下摆采用了压褶设计，增加了裙子的立体感，让整件裙子更加飘逸。裙子的领型为圆领，显得非常优雅。而裙子的版型设计则非常显瘦，让穿着者能够展现出优雅的身材曲线。总的来说，这是一款非常具有文艺风格、简约风格和显瘦效果的连衣裙。


In [5]:
model = PeftModel.from_pretrained(base_model, peft_model_path)
response, history = model.chat(tokenizer=tokenizer, query=input_text)
print(f'ChatGLM3-6B 微调后: \n{response}')

ChatGLM3-6B 微调后: 
简约的圆领设计，时尚百搭，凸显出女性柔美的气质，凸显出优雅动人的气质。精致印花图案，凸显出浪漫的文艺气息，凸显出女性优雅动人的气质。


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 加载原模型和tokenizer
model_id = "/root/autodl-fs/ZhipuAI/chatglm3-6b"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)

# 加载微调后的模型
peft_model_id = f"models/demo/{model_id}"
peft_model = PeftModel.from_pretrained(model, peft_model_id, trust_remote_code=True)

# 将微调后的权重合并到原模型
model = peft_model.merge_and_unload()

# 保存合并后的模型
model.save_pretrained("merged_model_directory")


The repository for /root/autodl-fs/ZhipuAI/chatglm3-6b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co//root/autodl-fs/ZhipuAI/chatglm3-6b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


The repository for /root/autodl-fs/ZhipuAI/chatglm3-6b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co//root/autodl-fs/ZhipuAI/chatglm3-6b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y
The repository for /root/autodl-fs/ZhipuAI/chatglm3-6b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co//root/autodl-fs/ZhipuAI/chatglm3-6b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 加载原模型和tokenizer
model_id = "/root/autodl-fs/ZhipuAI/chatglm3-6b"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.save_pretrained("merged_model_directory")

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


('merged_model_directory/tokenizer_config.json',
 'merged_model_directory/special_tokens_map.json',
 'merged_model_directory/tokenizer.model',
 'merged_model_directory/added_tokens.json')

In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value




# # from huggingface_hub import model_upload

# # 你的 Huggingface Hub 的用户名和模型名
# username = "snowfly"
# model_name = "chatglm3-6B-QLoRA-adgen"
# model_id = "merged_model_directory"

# # 上传模型
# # upload_model = model_upload(
# #     repo_id=f"{username}/{model_name}",
# #     commit_message="Upload merged model",
# #     push_to_hub=True,
# # )

# from huggingface_hub import HfApi

# # 初始化Huggingface API
# api = HfApi()

# # 上传模型和tokenizer
# # 这将创建一个公开的仓库，您可以通过设置`private=True`来创建一个私有的仓库
# api.upload_folder(
#     repo_id=f"{username}/{model_name}",
#     folder_path=model_id,
#     repo_type="model",
#     # private=False,
#     # create_repo=True,
# )

# print(f"模型已成功上传到Huggingface，地址为: https://huggingface.co/{username}/{model_id}")


from huggingface_hub import Repository

# 创建一个Repository对象
repo = Repository(
    "merged_model_directory",  # 你的模型目录
    clone_from="snowfly/chatglm3-6B-QLoRA-adgen",  # 你的Huggingface账户名和模型ID
    # use_auth_token="your-huggingface-token"  # 你的Huggingface token
)

# 使用git推送你的模型到Huggingface
repo.push_to_hub()



/root/miniconda3/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)


OSError: Looks like you do not have git-lfs installed, please install. You can install from https://git-lfs.github.com/. Then run `git lfs install` (you only have to do this once).

In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value


from huggingface_hub import HfApi, notebook_login

# 登录到 Hugging Face（在 Jupyter 笔记本中使用）
notebook_login()

# 或者，如果你已经通过命令行登录，你可以跳过登录步骤

# 创建 HfApi 实例
api = HfApi()

# 指定要上传的文件夹路径、仓库ID和仓库类型
# folder_path = "path_to_your_folder"  # 你的模型文件夹路径
folder_path = "merged_model_directory"
# repo_id = "your_username/your_repo_name"  # 例如 "snowfly/hihi"
repo_id = "snowfly/chatglm3-6B-QLoRA-adgen"
repo_type = "model"  # 仓库类型

# 如果仓库不存在，先创建仓库
api.create_repo(repo_id=repo_id, repo_type=repo_type,
                private=False)  # 根据需要设置 private 参数

# 上传文件夹到 Hugging Face Hub
api.upload_folder(
    folder_path=folder_path,
    repo_id=repo_id,
    repo_type=repo_type,
)

print(f"Folder {folder_path} has been uploaded to Hugging Face Hub successfully!")

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-6669cc67-0bbffd025c1a86511445c525;80d915cb-fd68-4be2-bd1b-68c3bc98aabc)

Invalid username or password.